In [4]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".joblib"):
            print(os.path.join(dirname, filename))

/kaggle/input/models/danilzhukovv/ecup-baseline-logreg-l12/scikitlearn/default/1/baseline_logreg_l12.joblib


In [1]:
import os,json,gc,time,random
os.environ["TOKENIZERS_PARALLELISM"]="false"
from pathlib import Path
from collections import Counter
import numpy as np,pandas as pd,pyarrow.parquet as pq,torch
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_linear_schedule_with_warmup

BASE="/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"
ITEMS=f"{BASE}/items.parquet"
LLM=f"{BASE}/matches_llm.parquet"
POLYGON_DIR="/kaggle/input/datasets/w4stdd/2m-parquet"
POLYGON=f"{POLYGON_DIR}/llm_sample_2m.parquet"
RESULT="/kaggle/working/ablation_results.json"
MODEL="cointegrated/rubert-tiny2"
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

for p in [ITEMS,LLM,POLYGON]:
    assert Path(p).exists(),f"Файл не найден: {p}"
assert torch.cuda.is_available(),"GPU не включена"
DEV=torch.device("cuda:0")
GPUS=list(range(torch.cuda.device_count()))
print("GPU:",[torch.cuda.get_device_name(i) for i in GPUS])
print("Polygon:",POLYGON)

EXPS=[
    {"name":"tiny_anchor","confw":False,"catbal":False},
    {"name":"tiny_confw","confw":True,"catbal":False},
    {"name":"tiny_catbal","confw":False,"catbal":True},
    {"name":"tiny_confw_catbal","confw":True,"catbal":True},
]

ml=pd.read_parquet(LLM,columns=["id1","id2","target"])
parent={}
def find(x):
    p=parent.setdefault(x,x)
    while p!=parent[p]:
        parent[p]=parent[parent[p]]; p=parent[p]
    parent[x]=p
    return p

for a,b in zip(ml.id1.values,ml.id2.values):
    a,b=find(a),find(b)
    if a!=b: parent[b]=a

comp=np.fromiter((find(x) for x in ml.id1.values),np.int64,len(ml))
rng=np.random.RandomState(13); groups=np.unique(comp)
val_groups=set(groups[rng.rand(len(groups))<.03].tolist())
mask=np.fromiter((x in val_groups for x in comp),bool,len(comp))
val=ml[mask].copy()
val=val[(val.target<=.2)|(val.target>=.8)]
val["target"]=(val.target>=.5).astype(np.int8)
del ml,parent,comp,mask,val_groups,groups
gc.collect()

train=pd.read_parquet(POLYGON,columns=["id1","id2","target"])
need=set(train.id1)|set(train.id2)|set(val.id1)|set(val.id2)
raw={}

def parse_attrs(x):
    try:
        d=json.loads(x) if isinstance(x,str) else x if isinstance(x,dict) else {}
        return {str(k).lower():str(v) for k,v in d.items() if v}
    except:
        return {}

pf=pq.ParquetFile(ITEMS)
for batch in pf.iter_batches(columns=["id","name","attributes","category"],batch_size=500_000):
    d=batch.to_pandas()
    d=d[d.id.isin(need)]
    for i,n,a,c in d.itertuples(index=False,name=None):
        raw[i]=(str(n) if n is not None else "",parse_attrs(a),
                str(c) if c is not None else "")

val["category"]=[raw[x][2] for x in val.id1]
fast=val.sample(min(60_000,len(val)),random_state=0).reset_index(drop=True)
print(f"Полигон: {len(train):,}; holdout: {len(val):,}; товаров: {len(raw):,}")

KEYS=["бренд","артикул","партномер","oem","код","модель","размер",
      "цвет","объем","обьем","вес","тип","материал","количество"]

def make_text(item):
    name,attrs,_=item; chosen=[]; used=set()
    for wanted in KEYS:
        for k,v in attrs.items():
            if wanted in k and k not in used:
                chosen.append((k,v)); used.add(k)
    chosen.extend((k,v) for k,v in attrs.items() if k not in used)
    return f"{name} | "+(" ; ".join(f"{k}:{v}" for k,v in chosen)[:260])

texts={i:make_text(item) for i,item in raw.items()}

class PairDS(Dataset):
    def __init__(self,df,weights=None):
        self.a=df.id1.to_numpy()
        self.b=df.id2.to_numpy()
        self.y=df.target.to_numpy(np.float32)
        self.weights=weights
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        w=1.0 if self.weights is None else self.weights[i]
        return texts[self.a[i]],texts[self.b[i]],self.y[i],w

def macro_ap(df,pred):
    z=df[["category","target"]].copy()
    z["prediction"]=pred
    return float(np.mean([
        average_precision_score(g.target,g.prediction)
        for _,g in z.groupby("category")
    ]))

@torch.inference_mode()
def predict(model,tok,df):
    model.eval(); predictions=[]
    def collate(rows):
        return tok(
            [x[0] for x in rows],[x[1] for x in rows],
            padding=True,truncation=True,max_length=160,return_tensors="pt"
        )
    loader=DataLoader(PairDS(df),batch_size=512,shuffle=False,
                      num_workers=0,collate_fn=collate)
    for enc in loader:
        enc={k:v.to(DEV) for k,v in enc.items()}
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(**enc).logits.squeeze(-1)
        predictions.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(predictions)

def run_experiment(cfg):
    started=time.time()
    tok=AutoTokenizer.from_pretrained(MODEL)
    base_model=AutoModelForSequenceClassification.from_pretrained(
        MODEL,num_labels=1
    ).to(DEV)
    model=(torch.nn.DataParallel(base_model,device_ids=GPUS)
           if len(GPUS)>1 else base_model)

    weights=np.ones(len(train),dtype=np.float32)
    if cfg["catbal"]:
        counts=Counter(raw[x][2] for x in train.id1)
        median=float(np.median(list(counts.values())))
        category_weights={
            c:float(np.clip((median/n)**.5,.65,2.0))
            for c,n in counts.items()
        }
        norm=sum(counts[c]*category_weights[c] for c in counts)/sum(counts.values())
        category_weights={c:w/norm for c,w in category_weights.items()}
        weights=np.array([
            category_weights.get(raw[x][2],1.0) for x in train.id1
        ],dtype=np.float32)

    def collate(rows):
        a,b,y,w=map(list,zip(*rows))
        enc=tok(a,b,padding=True,truncation=True,max_length=160,return_tensors="pt")
        y=torch.tensor(y,dtype=torch.float32)
        w=torch.tensor(w,dtype=torch.float32)
        if cfg["confw"]:
            w*=.75+.25*(2*y-1).abs()
        return enc,y,w

    loader=DataLoader(
        PairDS(train,weights),batch_size=256,shuffle=True,drop_last=True,
        num_workers=0,pin_memory=True,collate_fn=collate
    )
    optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=.01)
    scheduler=get_linear_schedule_with_warmup(
        optimizer,len(loader)//30,len(loader)
    )
    scaler=torch.amp.GradScaler("cuda")
    model.train(); timer=time.time(); seen=0

    for enc,y,w in loader:
        enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}
        y=y.to(DEV,non_blocking=True)
        w=w.to(DEV,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(**enc).logits.squeeze(-1)
            losses=F.binary_cross_entropy_with_logits(
                logits,y,reduction="none"
            )
            loss=(losses*w).sum()/w.sum().clamp_min(1e-6)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        seen+=len(y)

        if seen%(256*800)<256:
            speed=seen/(time.time()-timer)
            print(
                f"{cfg['name']}: {seen:,}/{len(train):,}, "
                f"loss={loss.item():.4f}, {speed:.0f} pair/s",
                flush=True
            )

    fast_score=macro_ap(fast,predict(model,tok,fast))
    full_score=macro_ap(val,predict(model,tok,val))
    result={
        **cfg,
        "fast_macro":round(fast_score,4),
        "full_macro":round(full_score,4),
        "minutes":round((time.time()-started)/60,1)
    }

    del model,base_model,tok,loader,optimizer,scheduler
    gc.collect(); torch.cuda.empty_cache()
    return result

results=json.load(open(RESULT)) if Path(RESULT).exists() else []
finished={r["name"] for r in results}

for cfg in EXPS:
    if cfg["name"] in finished:
        continue
    print(f"\n=== {cfg['name']} ===",flush=True)
    try:
        result=run_experiment(cfg)
        results.append(result)
        with open(RESULT,"w") as f:
            json.dump(results,f,ensure_ascii=False,indent=2)
        print(pd.DataFrame(results)[
            ["name","confw","catbal","fast_macro","full_macro","minutes"]
        ].to_string(index=False),flush=True)
    except Exception as error:
        print(
            f"{cfg['name']} FAILED: {type(error).__name__}: {error}",
            flush=True
        )
        gc.collect(); torch.cuda.empty_cache()

print(f"\nГОТОВО. Результат сохранён: {RESULT}")

GPU: ['Tesla T4', 'Tesla T4']
Polygon: /kaggle/input/datasets/w4stdd/2m-parquet/llm_sample_2m.parquet
Полигон: 2,170,867; holdout: 191,555; товаров: 2,579,865

=== tiny_anchor ===


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

tiny_catbal: 204,800/2,170,867, loss=0.5185, 835 pair/s
tiny_catbal: 409,600/2,170,867, loss=0.4836, 836 pair/s
tiny_catbal: 614,400/2,170,867, loss=0.4448, 839 pair/s
tiny_catbal: 819,200/2,170,867, loss=0.4761, 840 pair/s
tiny_catbal: 1,024,000/2,170,867, loss=0.4595, 840 pair/s
tiny_catbal: 1,228,800/2,170,867, loss=0.4896, 840 pair/s
tiny_catbal: 1,433,600/2,170,867, loss=0.4202, 839 pair/s
tiny_catbal: 1,638,400/2,170,867, loss=0.5013, 837 pair/s
tiny_catbal: 1,843,200/2,170,867, loss=0.4409, 837 pair/s
tiny_catbal: 2,048,000/2,170,867, loss=0.4132, 836 pair/s
       name  confw  catbal  fast_macro  full_macro  minutes
tiny_anchor  False   False      0.5666      0.5662     46.8
 tiny_confw   True   False      0.5629      0.5613     47.3
tiny_catbal  False    True      0.5467      0.5408     46.8

=== tiny_confw_catbal ===


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

tiny_confw_catbal: 204,800/2,170,867, loss=0.5009, 851 pair/s
tiny_confw_catbal: 409,600/2,170,867, loss=0.5111, 849 pair/s
tiny_confw_catbal: 614,400/2,170,867, loss=0.4692, 851 pair/s
tiny_confw_catbal: 819,200/2,170,867, loss=0.4113, 852 pair/s
tiny_confw_catbal: 1,024,000/2,170,867, loss=0.4071, 852 pair/s
tiny_confw_catbal: 1,228,800/2,170,867, loss=0.4894, 854 pair/s
tiny_confw_catbal: 1,433,600/2,170,867, loss=0.4280, 855 pair/s
tiny_confw_catbal: 1,638,400/2,170,867, loss=0.3844, 855 pair/s
tiny_confw_catbal: 1,843,200/2,170,867, loss=0.4763, 856 pair/s
tiny_confw_catbal: 2,048,000/2,170,867, loss=0.4620, 856 pair/s
             name  confw  catbal  fast_macro  full_macro  minutes
      tiny_anchor  False   False      0.5666      0.5662     46.8
       tiny_confw   True   False      0.5629      0.5613     47.3
      tiny_catbal  False    True      0.5467      0.5408     46.8
tiny_confw_catbal   True    True      0.5758      0.5705     45.6

ГОТОВО. Результат сохранён: /kaggle/w